안되면 이거 설치하기

pip install python-certifi-win32

In [7]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from ddgs import DDGS

load_dotenv()

llm = ChatOpenAI(model="gpt-4.1-nano")

# import os
# # 파이썬의 표준 라이브러리( 예: urllib, ssl )가 참조할 인증서 파일 경로를 설정함
# # "r"은 Raw String으로, 역슬래시(\)를 경로 기호로 그대로 인식하게 함
# os.environ["SSL_CERT_FILE"] = r"C:\cert\cacert.pem"

# # curl 기반의 라이브러리나 일부 하위 시스템이 참조할 인증서 묶음(Bundle) 경로를 설정함
# # 보안 네트워크(방화벽) 환경에서 인증서 오류를 해결하기 위해 자주 사용됨
# os.environ["CURL_CA_BUNDLE"] = r"C:\cert\cacert.pem"

# ddgs 웹 검색 tool
@tool
def web_search(query: str) -> str:
    """웹에서 최신 정보를 검색함"""

    # DuckDuckGo 검색 실행
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_result=3)    # max_result=3 -> 상위 3개 결과

            if not results:
                return "검색 결과가 없습니다."
            
            output = []
            
            for r in results:
                
                output.append(f"{r['title']} - {r['href']}")

            return "\n".join(output)
        
    except Exception as e:
        return f"검색 오류 : {e}"
    
# Agent 생성
agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="최신 정보가 필요하면 반드시 web_search 도구를 사용하세요."
)

# 실행
result = agent.invoke(
    {
        "messages": [
            {"role":"user", "content" : "도쿄 여행 3박 4일 일정을 추천해줘"}
        ]
    }
)

# 결과 출력
print(result)

{'messages': [HumanMessage(content='도쿄 여행 3박 4일 일정을 추천해줘', additional_kwargs={}, response_metadata={}, id='0ecf4dcf-622e-423f-8e84-7b96529d66af'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 74, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_cd97a2dd68', 'id': 'chatcmpl-DMq6JJGs7i8VF2OCDOUrpODqb0wgZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1ea8-a9af-7b00-aa7f-8405a3256cca-0', tool_calls=[{'name': 'web_search', 'args': {'query': '도쿄 3박 4일 여행 일정 추천'}, 'id': 'call_BjQfzPCWjYTHClJHD2AaN5st', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'out

In [8]:
from openai import OpenAI
import json

client = OpenAI()

prompt = """
서울 2박 3일 여행 일정을 짜주세요.
반드시 아래 JSON 형식으로만 응답하세요. 다른 텍스트는 절대 포함하지 마세요.

{
    "user_message": "사용자에게 보여줄 전체 여행 일정 텍스트 (구체적인 활동, 음식, 이동수단 등)",
    "locations": [
        {
            "day": 1,
            "name": "장소명",
            "address": "주소 (Nominatim 검색용)",
            "description": "이 장소에서 할 것"
        }
    ]
}

locations의 name과 address는 반드시 Nominatim으로 검색 가능한 실제 존재하는 장소여야 합니다.
"""

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_object"}  # JSON 응답 강제
)

result = json.loads(response.choices[0].message.content)

user_message = result["user_message"]   # 사용자에게 보여줄 텍스트
locations = result["locations"]          # 지도에 찍을 장소 리스트

In [10]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

geolocator = Nominatim(user_agent="travel_planner")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # 초당 1회 제한 준수

for loc in locations:
    result = geocode(loc["address"])
    if result:
        loc["lat"] = result.latitude
        loc["lng"] = result.longitude
    else:
        loc["lat"] = None
        loc["lng"] = None
        print(f"⚠️ 위경도 변환 실패: {loc['name']}")

# 변환 실패한 장소 제거
valid_locations = [loc for loc in locations if loc["lat"] is not None]


In [11]:
print(valid_locations)

[{'day': 1, 'name': '이태원', 'address': '서울특별시 용산구 이태원동', 'description': '다양한 문화와 음식을 체험할 수 있는 트렌디한 거리.', 'lat': 37.53842, 'lng': 126.992}, {'day': 1, 'name': 'N 서울타워', 'address': '서울특별시 용산구 남산공원길 105', 'description': '서울의 전경을 한눈에 볼 수 있는 명소.', 'lat': 37.5512692, 'lng': 126.9882959}, {'day': 2, 'name': '경복궁', 'address': '서울특별시 종로구 사직로 161', 'description': '조선 시대의 궁궐을 탐방하며 역사 배우기.', 'lat': 37.5759194, 'lng': 126.9768156}, {'day': 2, 'name': '북촌 한옥마을', 'address': '서울특별시 종로구 계동길 37', 'description': '전통 한옥 건축물 감상하기.', 'lat': 37.579046, 'lng': 126.9863882}, {'day': 2, 'name': '인사동', 'address': '서울특별시 종로구 인사동길', 'description': '전통 찻집과 기념품 쇼핑을 즐길 수 있는 거리.', 'lat': 37.5735445, 'lng': 126.9855646}, {'day': 3, 'name': '동대문디자인플라자', 'address': '서울특별시 중구 을지로 281', 'description': '현대적인 건축물을 감상하고 쇼핑하기.', 'lat': 37.5670686, 'lng': 127.009899}, {'day': 3, 'name': '한강 공원', 'address': '서울특별시 영등포구 여의도동', 'description': '자전거 타기 및 여유로운 산책.', 'lat': 37.5284646, 'lng': 126.9300522}]


In [12]:
print(user_message)

서울 2박 3일 여행 일정입니다! 

첫째 날, 서울에 도착하여 먼저 이태원을 방문해보세요. 힙하고 트렌디한 거리에서 다양한 음식과 문화를 즐길 수 있습니다. 저녁에는 남산타워로 올라가 서울의 아름다운 야경을 감상하세요. 

둘째 날, 아침에는 경복궁을 둘러보세요. 전통적인 한국 건축물을 감상하며 조선 역사를 배우실 수 있습니다. 그 후 북촌 한옥마을로 이동하여 한옥들의 아름다움을 느껴보세요. 점심은 인근 삼청동에서 전통 한식을 드시고, 오후에는 인사동 거리에서 쇼핑과 전통 찻집에서 티타임을 즐기세요. 

셋째 날, 아침 일찍 동대문 디자인 플라자로 이동하여 현대적인 건축물을 구경하세요. 근처 동대문 시장에서 기념품 쇼핑을 한 후, 근처에서 점심을 드세요. 오후에는 한강에서 자전거를 타거나, 운치있는 카페에서 여유를 즐기며 서울 여행을 마무리하세요.
